# MobileNets — Efficient CNNs for Mobile Vision Applications (2017)

_Papers · Computer Vision_

**Split a normal convolution into a per-channel filter plus a 1x1 mixer, cutting compute 8-9x.**

---

This notebook is a practice scaffold for the **MobileNets — Efficient CNNs for Mobile Vision Applications (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- 0. Recompute the lesson's worked example: DK=3, M=3, N=4, DF=8. ---
def conv_cost(dk, m, n, df):  # standard conv multiply-adds (eqn 2)
    return dk*dk * m * n * df*df
def sep_cost(dk, m, n, df):   # depthwise (eqn 4) + pointwise 1x1 (eqn 5)
    return dk*dk * m * df*df  +  m * n * df*df

DK, M, N, DF = 3, 3, 4, 8
std = conv_cost(DK, M, N, DF)
dw  = DK*DK * M * DF*DF
pw  = M * N * DF*DF
sep = sep_cost(DK, M, N, DF)
print("worked example  standard =", std, " depthwise =", dw, " pointwise =", pw, " separable =", sep)
print("ratio sep/std   =", round(sep/std, 4), "   formula 1/N + 1/DK^2 =", round(1/N + 1/DK**2, 4))
# worked example  standard = 6912  depthwise = 1728  pointwise = 768  separable = 2496
# ratio sep/std   = 0.3611    formula 1/N + 1/DK^2 = 0.3611


# --- 1. The depthwise-separable block (built by hand). The 'groups=in_ch' is the whole trick. ---
class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # depthwise: each input channel filtered on its own (groups=in_ch), no channel mixing
        self.dw   = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        self.bn_d = nn.BatchNorm2d(in_ch)
        # pointwise: 1x1 conv that mixes channels in_ch -> out_ch
        self.pw   = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn_p = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.relu(self.bn_d(self.dw(x)))   # filter each channel
        x = self.relu(self.bn_p(self.pw(x)))   # then mix channels
        return x

# A matched FULL standard conv block (the ablation): one ordinary 3x3 conv instead of the split.
class StandardConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


# --- 2. Two tiny matched nets: separable vs full convolution (same widths/depth). ---
class TinyNet(nn.Module):
    def __init__(self, Block, n_classes=5):
        super().__init__()
        self.b1 = Block(3, 16)
        self.b2 = Block(16, 32)
        self.head = nn.Linear(32, n_classes)
    def forward(self, x):
        x = self.b2(self.b1(x))
        x = x.mean(dim=(2, 3))          # global average pool
        return self.head(x)

def n_params(m): return sum(p.numel() for p in m.parameters())


# --- 3. Toy 5-class image data (no download needed): each class is a noisy prototype pattern. ---
g = torch.Generator().manual_seed(1)
Nimg, C, H, W, K = 600, 3, 16, 16, 5
y = torch.randint(0, K, (Nimg,), generator=g)
proto = torch.randn(K, C, H, W, generator=g)
X = proto[y] + 0.7 * torch.randn(Nimg, C, H, W, generator=g)
Xtr, ytr, Xte, yte = X[:480], y[:480], X[480:], y[480:]

def train(net, epochs=120, lr=0.08):
    torch.manual_seed(0)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    lf  = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train(); opt.zero_grad(); loss = lf(net(Xtr), ytr); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item()

sep_net = TinyNet(DepthwiseSeparable)
std_net = TinyNet(StandardConv)

# Multiply-adds of the two conv stages on a 16x16 map (padding keeps DF=16).
DFm = 16
std_flops = conv_cost(3, 3, 16, DFm) + conv_cost(3, 16, 32, DFm)
sep_flops = sep_cost(3, 3, 16, DFm)  + sep_cost(3, 16, 32, DFm)

acc_sep = train(sep_net)
acc_std = train(std_net)

print("\n              params      conv mult-adds   test acc")
print("separable   %7d      %12d     %.3f" % (n_params(sep_net), sep_flops, acc_sep))
print("full conv   %7d      %12d     %.3f" % (n_params(std_net), std_flops, acc_std))
print("reduction   %5.1fx fewer  %9.1fx fewer" % (n_params(std_net)/n_params(sep_net),
                                                  std_flops/sep_flops))
# separable ~5x fewer params, ~7x fewer mult-adds, accuracy close to the full-conv net.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does a depthwise-separable net reach near full-conv accuracy at a fraction of the params/FLOPs?_

In [ ]:
import torch, torch.nn as nn

torch.manual_seed(0)
g = torch.Generator().manual_seed(1)
N, C, H, W, K = 600, 3, 16, 16, 5
y = torch.randint(0, K, (N,), generator=g)
proto = torch.randn(K, C, H, W, generator=g)
X = proto[y] + 0.7 * torch.randn(N, C, H, W, generator=g)
Xtr, ytr, Xte, yte = X[:480], y[:480], X[480:], y[480:]

def conv_cost(dk, m, n, df): return dk*dk*m*n*df*df            # standard (eqn 2)
def sep_cost(dk, m, n, df):  return dk*dk*m*df*df + m*n*df*df  # depthwise + pointwise (eqn 5)

class Sep(nn.Module):   # depthwise (groups=in) + pointwise 1x1, each with BN + ReLU
    def __init__(s, ci, co):
        super().__init__()
        s.dw=nn.Conv2d(ci,ci,3,padding=1,groups=ci,bias=False); s.bd=nn.BatchNorm2d(ci)
        s.pw=nn.Conv2d(ci,co,1,bias=False); s.bp=nn.BatchNorm2d(co)
    def forward(s, x):
        x=torch.relu(s.bd(s.dw(x))); return torch.relu(s.bp(s.pw(x)))

class Std(nn.Module):   # one ordinary 3x3 conv + BN + ReLU (the ablation)
    def __init__(s, ci, co):
        super().__init__(); s.c=nn.Conv2d(ci,co,3,padding=1,bias=False); s.b=nn.BatchNorm2d(co)
    def forward(s, x): return torch.relu(s.b(s.c(x)))

class Net(nn.Module):
    def __init__(s, Block):
        super().__init__(); s.b1=Block(3,16); s.b2=Block(16,32); s.head=nn.Linear(32,K)
    def forward(s, x): return s.head(s.b2(s.b1(x)).mean(dim=(2,3)))

def nparams(m): return sum(p.numel() for p in m.parameters())
def train(net, epochs=120, lr=0.08):
    torch.manual_seed(0)
    opt=torch.optim.SGD(net.parameters(),lr=lr,momentum=0.9,weight_decay=1e-4)
    lf=nn.CrossEntropyLoss(); curve=[]
    for e in range(epochs):
        net.train(); opt.zero_grad(); loss=lf(net(Xtr),ytr); loss.backward(); opt.step()
        if e%8==0 or e==epochs-1:
            net.eval()
            with torch.no_grad(): curve.append((e, round((net(Xte).argmax(1)==yte).float().mean().item(),3)))
    return curve

sep, std = Net(Sep), Net(Std)
c_sep, c_std = train(sep), train(std)
DF=16
sf = conv_cost(3,3,16,DF)+conv_cost(3,16,32,DF)
pf = sep_cost(3,3,16,DF)+sep_cost(3,16,32,DF)
print("params  std=%d sep=%d  (%.1fx fewer)" % (nparams(std), nparams(sep), nparams(std)/nparams(sep)))
print("flops   std=%d sep=%d  (%.1fx fewer)" % (sf, pf, sf/pf))
print("full-conv acc curve:  ", c_std)
print("separable acc curve:  ", c_sep)
# params std=5301 sep=1030 (5.1x fewer); flops std=1290240 sep=187136 (6.9x fewer)
# full conv reaches 1.000, separable reaches 0.942 -- small accuracy drop, big cost saving.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working tiny MobileNet whose blocks are depthwise-separable. Replace
            each separable block with a single standard $3\times3$ convolution (same in/out channels), keeping
            everything else identical, and retrain. What happens to the parameter count, the compute, and the
            test accuracy &mdash; and what does the comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap only the block type: each DepthwiseSeparable(c_in, c_out) becomes one nn.Conv2d(c_in, c_out, 3, padding=1) + BatchNorm + ReLU. Keep depth, widths, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; standard vs separable convolution &mdash; so any difference is attributable to it._
- Print parameters and multiply-adds for both nets. — _The separable net should have several times fewer of each, matching the $\tfrac1N+\tfrac1{D_K^2}$ prediction._
- Train both and compare test accuracy. — _The separable net should land close to the full-conv net &mdash; the paper's "small reduction in accuracy" for a large cost saving._

**Answer:** The full-convolution net has several times more parameters and multiply-adds, yet only
                 a small accuracy edge over the depthwise-separable net. In our run the separable net used
                 ~5x fewer parameters and ~7x fewer convolution multiply-adds while reaching ~0.94 vs ~1.00 test
                 accuracy. Since the two nets are identical except for the convolution type, this isolates the
                 factorization as the cause: most of the cost of a standard convolution is buying very little
                 accuracy, exactly the trade Table 4 reports. The CODEVIZ panel shows this contrast.

</details>

**Problem 2.** A $3\times3$ convolution deep in a network turns $M=128$ channels into $N=256$ channels on a
            $14\times14$ feature map. Compute the standard cost, the separable cost, and the ratio, and say how
            close it is to the "9x" headline.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Standard (eqn 2): $3\cdot3\cdot128\cdot256\cdot14\cdot14 = 9\cdot128\cdot256\cdot196$. — _Plug $D_K=3, M=128, N=256, D_F=14$ into eqn 2._
- Separable (eqn 5): depthwise $3\cdot3\cdot128\cdot196$ plus pointwise $128\cdot256\cdot196$. — _Eqn 5 is the depthwise cost (no $N$) plus the pointwise $1\times1$ cost._
- Ratio (the shortcut): $\tfrac1N+\tfrac1{D_K^2} = \tfrac1{256}+\tfrac19 \approx 0.0039 + 0.1111 = 0.115$. — _The cancellation means you never need the big products &mdash; the ratio depends only on $N$ and $D_K$._

**Answer:** The ratio is $\tfrac1{256}+\tfrac19 \approx 0.115$, i.e. the separable version costs about
                 11.5% of the standard one &mdash; roughly 8.7x cheaper, right in the paper's
                 "8 to 9 times" range. Because $N=256$ is large, the $\tfrac1N$ term is nearly negligible and the
                 ratio is dominated by $\tfrac1{D_K^2}=\tfrac19$. Concretely the standard cost is about
                 $57.8$M and the separable cost about $6.7$M multiply-adds.

</details>

**Problem 3.** Your worked example had $D_K=3, M=3, N=4, D_F=8$, giving standard cost 6912 and separable cost 2496
            (ratio 0.3611). Why is this saving so much smaller than the 9x quoted for the rest of the network,
            and what single change to the numbers would push the ratio toward $\tfrac19$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the ratio: $\tfrac1N+\tfrac1{D_K^2} = \tfrac14+\tfrac19$. The $\tfrac14$ term is from the small $N=4$. — _With few output channels, $\tfrac1N$ is large and dominates the ratio, capping the saving._
- Imagine raising $N$ to, say, 256 while keeping $D_K=3$: $\tfrac1{256}+\tfrac19 \approx 0.115$. — _As $N$ grows, $\tfrac1N\to0$ and the ratio falls to $\tfrac1{D_K^2}=\tfrac19$._
- Note $D_K$ is fixed at 3 in MobileNet, so the only free knob here is the channel count $N$. — _The kernel-size term $\tfrac1{D_K^2}$ is the floor; the channel term is what varies layer to layer._

**Answer:** Because $N=4$ is tiny, the $\tfrac1N=0.25$ term swamps the $\tfrac19$ term, so the ratio is
                 $0.3611$ (only ~2.8x cheaper) rather than ~$0.11$. Increasing the output-channel count $N$
                 drives $\tfrac1N\to0$ and pushes the ratio toward $\tfrac1{D_K^2}=\tfrac19$ &mdash; the 8-9x
                 saving. That is why the headline figure applies to the deep, wide layers of the network, not the
                 narrow first ones.

</details>